# Plotly Visual Ideas for GHG per Capita
Dataset usati:
- `df_panel.csv` per la dinamica annuale 2014-2023


In [11]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display


pio.templates.default = 'plotly_white'
pd.options.display.float_format = '{:,.2f}'.format


## Animated Scatter Timeline

Visualise how GHG emissions per capita and GDP per capita co-evolved for every country from 2014 to 2023, animating one year at a time so individual country trajectories are easy to follow.


In [12]:
df_panel = pd.read_csv(Path('df_panel.csv'))
year_order = sorted(df_panel['year'].dropna().astype(int).unique().tolist())
timeline_df = df_panel.dropna(subset=['gdp_per_capita_ppp', 'ghg_per_capita', 'population']).copy()
timeline_df['year'] = pd.Categorical(timeline_df['year'].astype(int), categories=year_order, ordered=True)
timeline_df = timeline_df.sort_values(['year', 'country_name_ghg']).copy()
gold_red_scale = [(0.0, '#f2c14e'), (1.0, '#c1121f')]
ghg_color_range = [timeline_df['ghg_per_capita'].min(), timeline_df['ghg_per_capita'].max()]

# 1. We'll use GDP on the X-axis to spread the bubbles out. 
fig = px.scatter(
    timeline_df,
    x='gdp_per_capita_ppp', 
    y='ghg_per_capita',
    animation_frame='year',
    animation_group='iso3',
    color='ghg_per_capita',
    size='population',
    size_max=45,
    hover_name='country_name_ghg',
    hover_data={
        'year': False,
        'ghg_per_capita': ':.2f',
        'population': ':,.0f',
        'gdp_per_capita_ppp': ':,.0f',
        'renewable_pct': ':.1f',
        'region': True,
    },
    category_orders={'year': year_order},
    color_continuous_scale=gold_red_scale,
    range_color=ghg_color_range,
    # Set fixed ranges so the axes don't jump during animation
    range_x=[0, timeline_df['gdp_per_capita_ppp'].max() * 1.05],
    range_y=[0, timeline_df['ghg_per_capita'].max() * 1.1],
    title='Evolution of GHG vs Wealth (2014-2023)',
    template='plotly_white'
)

# 2. Add a white border to bubbles to prevent them from "bleeding" into each other
fig.update_traces(marker=dict(line=dict(width=1, color='white')))

# 3. Clean up the layout and UI with an increased figure size
fig.update_layout(
    height=900,         
    width=1200,         
    xaxis_title="GDP per Capita (PPP)",
    yaxis_title="GHG Emissions per Capita",
    coloraxis_colorbar=dict(title='GHG per Capita'),
    font=dict(size=14),
    title_font=dict(size=22),
    margin=dict(l=50, r=50, t=100, b=100) 
)

# 4. Slow down the animation slightly for better readability
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400
if fig.layout.sliders:
    fig.layout.sliders[0].y = -0.12
    fig.layout.sliders[0].x = 0.1
    fig.layout.sliders[0].len = 0.82
    fig.layout.sliders[0].currentvalue.prefix = 'Year: '
    fig.layout.sliders[0].steps = sorted(fig.layout.sliders[0].steps, key=lambda step: int(step['label']))

display(fig)